# CCA-F 考试复习 Notebook — Tool Design MCP

本 notebook 按 CCA-F Domain 2 task 列表组织。每个任务都包含概念解释、可运行 Python 示例、考点总结和 2 道官方风格模拟题。


## 环境初始化

In [ ]:
import json
import os
import re
from dataclasses import dataclass
from typing import Any

print("环境就绪：本 notebook 只使用本地模拟，不需要 API key 或外部服务。")


## D2.1 — 设计包含 tools、resources、prompts 的 MCP Server

### 📖 概念解释

MCP Server 的边界不是“把模型可能需要的一切都暴露出去”，而是把能力分成三类 primitive：`tools` 执行动作并可能改变外部状态，`resources` 提供只读上下文，`prompts` 复用经过设计的提示模板。CCA-F 常考的判断点是：读信息不应伪装成动作工具，动作工具必须有清晰输入 schema、权限边界和失败语义。

工具调用失败时应返回结构化结果，例如 `isError=true`、错误类别、可读消息和是否可重试；空结果则应是成功响应中的空数据。传输方式也要匹配部署边界：本地编辑器或 CLI 侧车进程常用 `stdio`，远程共享服务更适合 SSE 或 streamable HTTP，并要考虑认证、并发和网络失败。


In [ ]:
# Task 2.1：MCP primitives、isError 与传输边界
# 这个示例用本地字典模拟 MCP server，不启动真实服务，便于考试复习时观察边界设计。

mcp_server = {
    "name": "repo-review-server",
    "transport": "stdio",  # 本地 agent 启动的子进程通常使用 stdio。
    "tools": {
        "run_unit_tests": {
            "description": "运行允许列表中的测试命令。输入：{'command': 'pytest tests/unit'}。失败时返回 isError=true。",
            "mutates_state": False,
        },
        "create_issue": {
            "description": "在追踪系统创建 issue。输入必须包含 title 和 severity；不用于查询既有 issue。",
            "mutates_state": True,
        },
    },
    "resources": {
        "repo://architecture": "只读架构说明；适合模型先读取上下文，不应设计成写操作。",
        "repo://coding-standards": "团队编码规范，只读。",
    },
    "prompts": {
        "api_review": "请从认证、输入验证、错误处理和可观测性审查 API handler。",
    },
}

ALLOWED_TEST_COMMANDS = {"pytest tests/unit", "pytest tests/mcp"}

def mcp_result(content: Any, *, is_error: bool = False, category: str | None = None, retryable: bool = False) -> dict[str, Any]:
    """构造工具返回值：错误、空结果和成功结果必须可区分。"""
    result = {"isError": is_error, "content": content}
    if is_error:
        result["error"] = {"category": category or "unknown", "retryable": retryable}
    return result


def run_unit_tests(args: dict[str, str]) -> dict[str, Any]:
    command = args.get("command", "")
    if command not in ALLOWED_TEST_COMMANDS:
        return mcp_result(
            {"message": "测试命令不在允许列表中", "allowed": sorted(ALLOWED_TEST_COMMANDS)},
            is_error=True,
            category="policy_violation",
            retryable=False,
        )
    return mcp_result({"command": command, "passed": True, "tests": 12})


def choose_transport(location: str, shared_by_team: bool) -> str:
    # 考试重点：stdio 适合本地进程；远程共享服务需要 HTTP/SSE 类传输和认证边界。
    if location == "local" and not shared_by_team:
        return "stdio"
    return "streamable_http_or_sse"

print(json.dumps(mcp_server, ensure_ascii=False, indent=2))
print("成功调用:", run_unit_tests({"command": "pytest tests/unit"}))
print("边界错误:", run_unit_tests({"command": "rm -rf /"}))
print("远程团队服务传输:", choose_transport("remote", shared_by_team=True))


### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 用 `tools` 表达可调用动作，用 `resources` 暴露只读上下文，用 `prompts` 封装可复用模板。
- 工具失败返回 `isError=true` 和结构化错误 metadata；空结果仍是成功响应。
- 把权限、输入 schema、幂等性和副作用写进工具边界。
- 本地侧车进程优先考虑 `stdio`；远程共享服务考虑 SSE/HTTP、认证、并发和网络故障。

**✗ 常见陷阱**

- 把文档、配置和模板全部做成工具，导致模型误以为它们会执行动作。
- 用普通字符串“failed”表示失败，使协调器无法判断是否重试。
- 把会写入外部系统的工具描述成“查询工具”。
- 认为传输方式只是语法差异，忽略本地/远程部署边界。


### 🧪 模拟题

**Q1.** 一个 MCP tool 调用超时，但协调器需要判断是否可以重试。最合适的返回是什么？

A) `isError=true`，并包含 `category`、`message`、`retryable` 等结构化字段  
B) 空数组，因为本次没有拿到数据  
C) 普通字符串 `timeout`，让模型自行解释  
D) 直接终止整个 agent 进程

> **答案：A**  
> **解析：** 结构化错误能区分“失败”“空结果”和“可重试失败”，这是 MCP 工具可靠性的核心。

**Q2.** 团队要把本地 CLI 使用的 MCP server 改造成多人共享的远程服务。以下哪项最符合 CCA-F 判断？

A) 继续只使用 `stdio`，因为协议 primitive 没有变化  
B) 改用 SSE/streamable HTTP 等远程传输，并补充认证、并发和网络错误处理  
C) 把所有 resources 改成 tools，方便远程调用  
D) 删除 `isError` 字段，避免暴露内部失败细节

> **答案：B**  
> **解析：** 传输选择必须跟部署边界一致；远程共享服务需要网络、认证和并发语义。


## D2.2 — 配置工具分配并管理工具选择可靠性

### 📖 概念解释

工具选择可靠性首先来自工具集设计，而不是更长的系统提示。一个 agent 通常应只拿到 4-5 个与当前职责高度相关的工具；工具越多、命名越相似、描述越泛化，模型越容易误选。工具描述应包括输入格式、何时使用、何时不要使用、边界条件和典型边缘情况。

当两个工具功能重叠时，优先通过名称和描述消歧，例如 `lookup_order_by_id` 与 `search_orders_by_customer`，而不是依赖模型猜测。对高风险工具还应加程序化校验：参数 schema、允许列表、确认步骤或人工升级。


In [ ]:
# Task 2.2：工具分配与工具选择可靠性
# 示例展示：小工具集、清晰描述、边缘情况和重叠工具消歧。

agent_tools = {
    "refund_agent": [
        "verify_customer_identity",
        "lookup_order_by_id",
        "calculate_refund_quote",
        "submit_refund_request",
        "escalate_refund_case",
    ],
    "catalog_agent": [
        "search_products",
        "get_product_by_sku",
        "compare_product_specs",
        "check_inventory_by_sku",
    ],
}

tool_catalog = {
    "verify_customer_identity": "输入 email 或 customer_id；只验证身份，不查询订单。",
    "lookup_order_by_id": "输入 order_id，如 ORD-12345；只查询单个订单，不按客户搜索。",
    "search_orders_by_customer": "输入 customer_id；返回客户订单列表，不验证身份。",
    "submit_refund_request": "输入 order_id、amount、reason；会创建退款请求，高风险操作。",
}

def candidate_tools(agent_name: str) -> list[str]:
    tools = agent_tools[agent_name]
    if len(tools) > 5:
        raise ValueError(f"{agent_name} 工具过多：{len(tools)}。建议拆分 agent 或收紧职责。")
    return tools


def choose_refund_tool(user_request: str) -> str:
    text = user_request.lower()
    has_order_id = bool(re.search(r"\bord-\d+\b", text))
    has_email = bool(re.search(r"[\w.-]+@[\w.-]+", text))

    if "refund" in text or "退款" in user_request:
        if has_order_id:
            return "calculate_refund_quote"  # 先报价/校验，再提交高风险动作。
        return "ask_for_order_id"
    if has_order_id:
        return "lookup_order_by_id"
    if has_email or "customer" in text or "客户" in user_request:
        return "verify_customer_identity"
    return "ask_clarifying_question"

examples = [
    "请查询订单 ORD-12345",
    "用 ada@example.com 验证客户身份",
    "我要给 ORD-88888 办退款",
    "查一下这个客户的订单",
]

print("refund_agent tools:", candidate_tools("refund_agent"))
for request in examples:
    print(f"{request} => {choose_refund_tool(request)}")


### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 每个 agent 保持 4-5 个高相关工具，职责清晰时再扩展。
- 工具名体现输入和用途，例如 `lookup_order_by_id`，避免 `get_data`。
- 描述写明输入格式、适用场景、反例、边缘情况和副作用。
- 对重叠工具用名称、描述和参数 schema 消歧；对高风险动作加确认或升级。

**✗ 常见陷阱**

- 给所有 agent 暴露全部工具，导致选择空间过大。
- 两个工具都叫“lookup”或描述都写“获取信息”。
- 遇到误选后只追加 few-shot，不修复工具边界。
- 让模型自己决定是否执行退款、删除、发送等有副作用操作。


### 🧪 模拟题

**Q1.** 一个 agent 经常在 `lookup_order_by_id` 和 `search_orders_by_customer` 之间误选。第一步最应该做什么？

A) 改进工具命名和描述，明确输入格式、适用场景和反例  
B) 把两个工具合并成 `get_information`  
C) 给 agent 更多数据库工具，让模型有更多选择  
D) 提高 temperature，让模型更灵活

> **答案：A**  
> **解析：** 工具误选通常先从边界和描述修复；合并或增加工具会扩大歧义。

**Q2.** 关于每个 agent 的工具数量，哪项最符合 CCA-F 风格？

A) 尽量给 20 个以上工具，避免 agent 无法完成任务  
B) 通常保持 4-5 个高相关工具，必要时按职责拆分 agent  
C) 工具数量不重要，只要系统提示足够长  
D) 只保留一个万能工具，所有逻辑写在自然语言里

> **答案：B**  
> **解析：** 小而明确的工具集能提高选择可靠性；职责过宽时应拆分或重新设计边界。


## D2.3 — 在 .mcp.json 中配置 MCP 并管理环境变量

### 📖 概念解释

项目级 `.mcp.json` 用来注册团队共享的 MCP servers，通常会被版本化到仓库中；个人实验或机器特定配置则属于用户级作用域。配置应写清 server 名称、`command`、`args`、`env` 和运行上下文。考试常考：server 注册位置决定谁能使用它，环境变量决定密钥如何在本地、CI/CD 和部署环境中注入。

密钥不应硬编码进 `.mcp.json`。推荐写成 `${ENV_VAR}` 形式，并在 CI/CD secret store、开发者 shell 或部署平台中提供真实值。验证配置时应检查 JSON 结构、必填字段、环境变量引用、scope 是否适合团队共享，以及 CI 环境是否缺少必要变量。


In [ ]:
# Task 2.3：.mcp.json、env vars、作用域与 CI/CD
# 示例只验证配置结构，不读取真实密钥；可以在没有 API key 的本地环境运行。

mcp_json_text = """
{
  "mcpServers": {
    "orders": {
      "scope": "project",
      "command": "python",
      "args": ["-m", "company_mcp.orders"],
      "env": {
        "ORDERS_API_BASE": "https://api.example.com",
        "ORDERS_API_KEY": "${ORDERS_API_KEY}"
      }
    },
    "local_notes": {
      "scope": "user",
      "command": "python",
      "args": ["-m", "company_mcp.notes"],
      "env": {}
    }
  }
}
"""

ENV_REF = re.compile(r"^\$\{[A-Z][A-Z0-9_]*\}$")
SECRET_HINT = re.compile(r"(sk-|api[_-]?key|token)", re.IGNORECASE)

def validate_mcp_config(config: dict[str, Any], *, ci: bool = False) -> list[str]:
    errors: list[str] = []
    servers = config.get("mcpServers")
    if not isinstance(servers, dict) or not servers:
        return ["缺少 mcpServers 或格式错误"]

    for name, server in servers.items():
        if server.get("scope") not in {"project", "user"}:
            errors.append(f"{name}: scope 必须是 project 或 user")
        if not server.get("command"):
            errors.append(f"{name}: 缺少 command")
        if not isinstance(server.get("args", []), list):
            errors.append(f"{name}: args 必须是数组")
        for key, value in server.get("env", {}).items():
            value = str(value)
            if SECRET_HINT.search(key) and not ENV_REF.match(value):
                errors.append(f"{name}: {key} 应通过 ${{ENV_VAR}} 注入，不能硬编码")
            if ci and ENV_REF.match(value):
                env_name = value[2:-1]
                if env_name not in os.environ:
                    errors.append(f"{name}: CI/CD 缺少环境变量 {env_name}")
    return errors

config = json.loads(mcp_json_text)
print(json.dumps(config, ensure_ascii=False, indent=2))
print("本地结构检查:", validate_mcp_config(config))
print("CI/CD 检查:", validate_mcp_config(config, ci=True))


### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 团队共享 server 放项目级 `.mcp.json`；个人实验放用户级配置。
- `command`、`args`、`env` 明确可审查，server 名称稳定。
- 密钥用 `${ENV_VAR}` 引用，由本地 shell、CI/CD secret store 或部署平台注入。
- CI/CD 要单独验证必需环境变量是否存在，不能假设开发者本机配置可用。

**✗ 常见陷阱**

- 把 API key、token 或私有 URL 直接提交到 `.mcp.json`。
- 只在个人配置里注册团队 server，导致其他人和 CI 无法复现。
- `.mcp.json` JSON 合法但缺少 `command`、`args` 或必要 `env`。
- 忽略 scope，把本地临时工具暴露成团队项目依赖。


### 🧪 模拟题

**Q1.** 团队希望所有开发者和 CI 都能使用同一个 MCP server。最佳配置方式是什么？

A) 在项目级 `.mcp.json` 注册 server，密钥用 `${ENV_VAR}` 从环境注入  
B) 把 server 命令写进每个人的 shell history  
C) 只在 README 中用自然语言描述启动方式  
D) 把 API key 直接写进 `.mcp.json`，避免 CI 配置 secret

> **答案：A**  
> **解析：** 项目级配置可共享和审查；秘密必须通过环境或 secret store 注入。

**Q2.** 本地运行正常，但 CI 中 MCP server 启动失败并提示缺少 `ORDERS_API_KEY`。最可能的修复是什么？

A) 在 CI/CD secret store 中配置 `ORDERS_API_KEY`，并确保 job 注入该环境变量  
B) 把真实 key 提交到仓库，保证所有环境一致  
C) 把 server scope 改成 `user`，让 CI 自动继承开发者本机配置  
D) 删除 `env` 字段，让 server 自行猜测密钥

> **答案：A**  
> **解析：** CI 不继承开发者本地 shell；需要显式配置 secret 和环境变量注入。
